# Smart Suggestion System (Upgrade & Accessory Guidance)


In [12]:
# Import essential libraries and load the cleaned dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display, Markdown, HTML

df = pd.read_csv("cleaned_dataset.csv")


In [13]:
# Convert purchased accessories into a list format for market analysis

# Convert included items to list
df['Items_List'] = df['Included_Items'].fillna('').apply(
    lambda x: [i.strip() for i in str(x).split('|') if i.strip() and i.strip().lower() != 'none']
)
transactions = df['Items_List'].tolist()


In [14]:
# Convert purchased accessories into a list format for market analysis

# Convert included items to list
df['Items_List'] = df['Included_Items'].fillna('').apply(
    lambda x: [i.strip() for i in str(x).split('|') if i.strip() and i.strip().lower() != 'none']
)
transactions = df['Items_List'].tolist()


In [15]:
# Encode the transaction lists into a one-hot tabular format

te = TransactionEncoder()
te_data = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_data, columns=te.columns_)

In [16]:
# Find patterns and generate association rules with 100% confidence

freq = apriori(df_trans, min_support=0.01, use_colnames=True)
rules = association_rules(freq, metric="confidence", min_threshold=1.0)
rules = rules[(rules["lift"] >= 1)]

In [17]:
# Display the discovered association rules in a detailed, untruncated table

display(Markdown("### 📊 Association Rules Metrics (Confidence 1.0)"))
if not rules.empty:
    pd.set_option('display.max_colwidth', None)  # Ensure full description is visible
    display_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
    display_rules['antecedents'] = display_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    display_rules['consequents'] = display_rules['consequents'].apply(lambda x: ', '.join(list(x)))
    display(display_rules)
else:
    print("No rules found with 1.0 confidence and lift > 1.1")

### 📊 Association Rules Metrics (Confidence 1.0)

,antecedents,consequents,support,confidence,lift
0,1 year Microsoft 365 basic subscription,Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00,0.162402,1.0,2.919540
1,Trendsetter Backpack,1 year Microsoft 365 basic subscription,0.011811,1.0,6.157576
2,Microsoft Xbox Game Pass 3 month,Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00,0.068898,1.0,2.919540
3,Trendsetter Backpack,Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00,0.011811,1.0,2.919540
4,"HP 310 15.6 Laptop Backpack worth ₹2,999.00, 1 year Microsoft 365 basic subscription",Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00,0.026575,1.0,2.919540
5,"1 year Microsoft 365 basic subscription, HP X Entry Backpack worth ₹1999.00",Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00,0.050197,1.0,2.919540
6,"Trendsetter Backpack, Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00",1 year Microsoft 365 basic subscription,0.011811,1.0,6.157576
7,"Trendsetter Backpack, 1 year Microsoft 365 basic subscription",Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00,0.011811,1.0,2.919540
8,Trendsetter Backpack,"Microsoft Office Home 2024 Lifetime Subscription worth ₹9199.00, 1 year Microsoft 365 basic subscription",0.011811,1.0,6.157576


## User Classification

| **User Type**      | **RAM Condition** | **Price Condition**      | **Category Meaning**                                |
| ------------------ | ----------------- | ------------------------ | --------------------------------------------------- |
| **Low-Level User** | RAM < 8GB         | Price ≤ ₹50,000 (approx) | Entry-level system with basic performance           |
| **Mid-Level User** | RAM ≥ 8GB         | ₹50,000 – ₹70,000        | Balanced system for general use and multitasking    |
| **Premium User**   | RAM ≥ 16GB        | Price > ₹70,000          | High-end system for advanced and professional tasks |


In [18]:
# Categorize users into Low, Mid, or Premium tiers based on hardware specs

def classify_user(row):
    """Classifies users based on hardware and price."""
    if row["RAM_GB"] >= 16 and row["Final_Price"] > 70000:
        return "Premium"
    elif row["RAM_GB"] >= 8:
        return "Mid"
    else:
        return "Low"


In [19]:
# Generate smart hardware upsell and accessory cross-sell recommendations
def smart_suggestion(laptop_idx):
    laptop = df.iloc[laptop_idx]
    user_tier = classify_user(laptop)
    
    raw_details = laptop['Included_Items']
    current_accessories = [
        item.strip() for item in str(raw_details).split('|')
        if item.strip() and item.strip().lower() != 'none'
    ]

    suggested_accessories = []
    if not rules.empty:
        for _, rule in rules.sort_values(by="confidence", ascending=False).iterrows():
            if set(rule['antecedents']).issubset(set(current_accessories)):
                for item in rule['consequents']:
                    if item not in current_accessories and item not in suggested_accessories:
                        suggested_accessories.append(item)
    
    if not suggested_accessories:
        suggested_accessories = ["HP Wireless Mouse (High Comfort)", "HP Premium Laptop Bag", "Multi-port USB-C Hub"]

    print(f"\n{'='*55}")
    print(f"💻 LAPTOP: {laptop['Name']}")
    print(f"⚙️ CONFIG: {laptop['RAM_GB']}GB RAM | {laptop['Storage_GB']}GB Storage | CPU: {laptop['Processor_Level']}")
    print(f"👤 USER TIER: {user_tier} {'💎' if user_tier == 'Premium' else '🚀'}")
    print(f"{'='*55}")

    if user_tier != "Premium":
        print("\n📈 [UP-SELL] HARDWARE OPTIMIZATION OPTIONS")
        if user_tier == "Low":
            print("• [URGENT] Upgrade RAM to 16GB for a 2x boost in performance!")
            print("• [UPGRADE] Switch to an NVMe SSD for lightning-fast boot times.")
        elif user_tier == "Mid" and laptop['RAM_GB'] < 16:
            print("• [PRO-TIP] 16GB RAM upgrade will help in heavy multitasking.")
    else:
        print("\n✨ YOUR SYSTEM IS TOP-TIER")
        print("• Your hardware meets all professional standards. Focus on accessories!")

    print("\n🎁 [CROSS-SELL] RECOMMENDED ACCESSORIES")
    for i, item in enumerate(suggested_accessories[:3], 1):
        print(f"{i}. {item}")
    
    print("-" * 55)

In [20]:
# Generate smart hardware upsell and accessory cross-sell recommendations
def smart_suggestion(laptop_idx):
    laptop = df.iloc[laptop_idx]
    user_tier = classify_user(laptop)
    
    raw_details = laptop['Included_Items']
    current_accessories = [
        item.strip() for item in str(raw_details).split('|')
        if item.strip() and item.strip().lower() != 'none'
    ]

    suggested_accessories = []
    if not rules.empty:
        for _, rule in rules.sort_values(by="confidence", ascending=False).iterrows():
            if set(rule['antecedents']).issubset(set(current_accessories)):
                for item in rule['consequents']:
                    if item not in current_accessories and item not in suggested_accessories:
                        suggested_accessories.append(item)
    
    if not suggested_accessories:
        suggested_accessories = ["HP Wireless Mouse (High Comfort)", "HP Premium Laptop Bag", "Multi-port USB-C Hub"]

    print(f"\n{'='*55}")
    print(f"💻 LAPTOP: {laptop['Name']}")
    print(f"⚙️ CONFIG: {laptop['RAM_GB']}GB RAM | {laptop['Storage_GB']}GB Storage | CPU: {laptop['Processor_Level']}")
    print(f"👤 USER TIER: {user_tier} {'💎' if user_tier == 'Premium' else '🚀'}")
    print(f"{'='*55}")

    if user_tier != "Premium":
        print("\n📈 [UP-SELL] HARDWARE OPTIMIZATION OPTIONS")
        if user_tier == "Low":
            print("• [URGENT] Upgrade RAM to 16GB for a 2x boost in performance!")
            print("• [UPGRADE] Switch to an NVMe SSD for lightning-fast boot times.")
        elif user_tier == "Mid" and laptop['RAM_GB'] < 16:
            print("• [PRO-TIP] 16GB RAM upgrade will help in heavy multitasking.")
    else:
        print("\n✨ YOUR SYSTEM IS TOP-TIER")
        print("• Your hardware meets all professional standards. Focus on accessories!")

    print("\n🎁 [CROSS-SELL] RECOMMENDED ACCESSORIES")
    for i, item in enumerate(suggested_accessories[:3], 1):
        print(f"{i}. {item}")
    
    print("-" * 55)

In [21]:
# Run representative test cases for each user tier to verify results
print("=== CASE 1: PREMIUM USER ===")
smart_suggestion(14) 

print("\n=== CASE 2: MID USER ===")
smart_suggestion(1) 

print("\n=== CASE 3: LOW USER ===")
smart_suggestion(133)  

=== CASE 1: PREMIUM USER ===

💻 LAPTOP: HP Victus 39.6 cm (15.6) Gaming Laptop 15-fb3185AX, Silver
⚙️ CONFIG: 24.0GB RAM | 1024GB Storage | CPU: Other
👤 USER TIER: Premium 💎

✨ YOUR SYSTEM IS TOP-TIER
• Your hardware meets all professional standards. Focus on accessories!

🎁 [CROSS-SELL] RECOMMENDED ACCESSORIES
1. HP Wireless Mouse (High Comfort)
2. HP Premium Laptop Bag
3. Multi-port USB-C Hub
-------------------------------------------------------

=== CASE 2: MID USER ===

💻 LAPTOP: HP Laptop 39.6 cm (15.6) 15-fc0500AU, Silver
⚙️ CONFIG: 8.0GB RAM | 512GB Storage | CPU: Other
👤 USER TIER: Mid 🚀

📈 [UP-SELL] HARDWARE OPTIMIZATION OPTIONS
• [PRO-TIP] 16GB RAM upgrade will help in heavy multitasking.

🎁 [CROSS-SELL] RECOMMENDED ACCESSORIES
1. HP Wireless Mouse (High Comfort)
2. HP Premium Laptop Bag
3. Multi-port USB-C Hub
-------------------------------------------------------

=== CASE 3: LOW USER ===

💻 LAPTOP: HP Chromebook x360 35.6 cm (14) Laptop 14b-cd0011TU, Silver
⚙️ CONFIG: 4

## 📊 Basic Business Analysis (Survey Population: 53)

This section overlays our recommendation rules onto the survey data to calculate potential profit using basic math:
1. **Opportunity** = Users who want the item but don't own it yet.
2. **Revenue** = (Opportunity) × (Conversion Rate 20%) × (Item Price)
3. **ROI** = (Total Revenue - Marketing Cost) / Marketing Cost

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Constants
POPULATION = 53
CONVERSION = 0.20
MKTG_COST = 100000

# --- 1. ACCESSORIES (CROSS-SELL) ---
acc_data = [
    ["Wireless Mouse", 19, 1200], # [Name, Opportunity Count, Price]
    ["Laptop Bag", 24, 2500],
    ["Laptop Stand", 26, 2000],
    ["Cooling Pad", 24, 1500],
    ["External Storage", 26, 5000]
]
df_acc = pd.DataFrame(acc_data, columns=["Item", "Opportunity", "Price"])
df_acc["Est. Buyers"] = (df_acc["Opportunity"] * CONVERSION).round(1)
df_acc["Revenue"] = (df_acc["Est. Buyers"] * df_acc["Price"]).astype(int)

# --- 2. HARDWARE UPGRADES (UPSELL) ---
up_data = [
    ["Processor Upgrade (i7)", 15, 12000],  # [Name, Survey Hits, Price]
    ["Battery Replacement", 13, 3000],
    ["RAM Upgrade (16GB)", 11, 4000],
    ["Storage (NVMe SSD)", 10, 5000],
    ["GPU Upgrade", 4, 25000]
]
df_up = pd.DataFrame(up_data, columns=["Item", "Hits", "Price"])
df_up["Est. Buyers"] = (df_up["Hits"] * CONVERSION).round(1)
df_up["Revenue"] = (df_up["Est. Buyers"] * df_up["Price"]).astype(int)

# --- FINAL SUMMARY ---
total_rev = df_acc["Revenue"].sum() + df_up["Revenue"].sum()
roi = (total_rev - MKTG_COST) / MKTG_COST * 100

display(Markdown("### 🛍️ Accessory Cross-Sell Potential"))
display(df_acc)

display(Markdown("### ⚡ Hardware Upsell Potential"))
display(df_up)

print(f"\n{'='*40}")
print(f"💰 TOTAL ESTIMATED REVENUE: ₹{total_rev:,}")
#print(f"💸 MARKETING COST:         ₹{MKTG_COST:,}")
print(f"📈 ESTIMATED ROI:           {roi:.1f}%")
print(f"{'='*40}")

if total_rev > MKTG_COST:
    print("✅ Result: PROJECTION IS PROFITABLE")
else:
    print(f"⚠️ Result: GAP OF ₹{MKTG_COST - total_rev:,} TO REACH BREAK-EVEN")

### 🛍️ Accessory Cross-Sell Potential

,Item,Opportunity,Price,Est. Buyers,Revenue
0,Wireless Mouse,19,1200,3.8,4560
1,Laptop Bag,24,2500,4.8,12000
2,Laptop Stand,26,2000,5.2,10400
3,Cooling Pad,24,1500,4.8,7200
4,External Storage,26,5000,5.2,26000


### ⚡ Hardware Upsell Potential

,Item,Hits,Price,Est. Buyers,Revenue
0,Processor Upgrade (i7),15,12000,3.0,36000
1,Battery Replacement,13,3000,2.6,7800
2,RAM Upgrade (16GB),11,4000,2.2,8800
3,Storage (NVMe SSD),10,5000,2.0,10000
4,GPU Upgrade,4,25000,0.8,20000



💰 TOTAL ESTIMATED REVENUE: ₹142,760
💸 MARKETING COST:         ₹100,000
📈 ESTIMATED ROI:           42.8%
✅ Result: PROJECTION IS PROFITABLE
